In [1]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from IPython.display import HTML, display 
from glob import glob
import ipywidgets as widgets
from IPython.display import clear_output
import numpy as np
from ipywidgets import Dropdown, HBox, VBox, Output, Button, HTML
import warnings

# Name of output directories
OUT_DIR = Path("drift_simple_report")
PLOTS_DIR = OUT_DIR / "plots"
assert PLOTS_DIR.exists(), "No se encontró drift_simple_report/plots. Ejecuta primero: python run_drift.py"
plt.rcParams.update({})  

### Functions

In [2]:
def Top5_table_graph(df_test: pd.DataFrame,
                     df_val: pd.DataFrame,
                     plots_dir: str = "drift_simple_report/plots",
                     img_test_name: str = "top5_test.png",
                     img_val_name: str = "top5_val.png"):
   

    # --- LIMPIEZA DE TABLAS ---
    for df in (df_test, df_val):
        for col in ("below_alpha", "drift_rate", "min_pvalue"):
            if col in df.columns:
                df.drop(columns=col, inplace=True)
        if "median_pvalue" in df.columns:
            df.rename(columns={"median_pvalue": "p_value"}, inplace=True)
        if "p_value" in df.columns:
            df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce").round(2)

    # --- Generar HTML de tablas con estilo SOLO en la tabla ---
    def _styled_table_html(df: pd.DataFrame) -> str:
        # Inyectamos estilos SOLO en la etiqueta <table>
        base = df.to_html(index=False, border=1)
        styled = base.replace(
            "<table ",
            "<table style='background:white; display:inline-block; "
            "border-collapse:collapse; border:1px solid #e5e7eb; "
            "box-shadow:0 1px 3px rgba(0,0,0,.08); "
            "font-family:ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial; "
            "font-size:14px' "
        )
        # Bordes y padding de celdas más limpios
        styled = styled.replace("<th>", "<th style='padding:6px 10px; border:1px solid #e5e7eb; text-align:left;'>")
        styled = styled.replace("<td>", "<td style='padding:6px 10px; border:1px solid #e5e7eb; text-align:left;'>")
        return styled

    left_tables_html = f"""
    <div style="display:flex; justify-content:flex-start; gap:40px; margin-top:10px;">
        <div style="text-align:left;">
            <h4 style="margin:4px 0 8px 0;">Top-5 Drift — Test</h4>
            {_styled_table_html(df_test)}
        </div>
        <div style="text-align:left;">
            <h4 style="margin:4px 0 8px 0;">Top-5 Drift — Val</h4>
            {_styled_table_html(df_val)}
        </div>
    </div>
    """
    display(HTML(left_tables_html))

    # --- IMÁGENES CENTRADAS ---
    plots_path = Path(plots_dir)
    test_img = plots_path / img_test_name
    val_img  = plots_path / img_val_name

    if test_img.exists() and val_img.exists():
        html_imgs = f"""
        <div style="display:flex; justify-content:center; gap:50px; margin-top:20px;">
            <div style="text-align:center;">
                <h4 style="color:white; margin:4px 0 8px 0;">Top-5 Drift — Test</h4>
                <img src="{test_img.as_posix()}" width="560">
            </div>
            <div style="text-align:center;">
                <h4 style="color:white; margin:4px 0 8px 0;">Top-5 Drift — Val</h4>
                <img src="{val_img.as_posix()}" width="560">
            </div>
        </div>
        """
        display(HTML(html_imgs))
    
def feature_image_viewer(plots_dir: str = "drift_simple_report/plots",
                         default_split: str = "test",
                         img_width: int = 700):
    """
    Visor interactivo con 2 imágenes FIJAS:
      - 1 histograma:  {feature}_hist_all.png
      - 1 p-values:    {feature}_pvalues_{split}.png
    >>> Nunca duplica: solo actualiza el contenido de los mismos widgets.
    """
    PLOTS_DIR = Path(plots_dir)

    # Descubrir features a partir de los archivos de histograma
    hist_files = glob(str(PLOTS_DIR / "*_hist_all.png"))
    features = sorted({Path(f).name.replace("_hist_all.png", "") for f in hist_files})
    if not features:
        print("No se encontraron histogramas en:", PLOTS_DIR)
        return

    # --- Widgets de control
    feat_dd  = widgets.Dropdown(options=features, description="Feature:", value=features[0])
    split_dd = widgets.Dropdown(options=[("test", "test"), ("val", "val")],
                                value=default_split, description="Split:")

    # --- Widgets de visualización (persistentes: no se “displayean” nuevos)
    title_html = widgets.HTML()
    hist_img   = widgets.Image(layout=widgets.Layout(width=f"{img_width}px"))
    pval_img   = widgets.Image(layout=widgets.Layout(width=f"{img_width}px"))
    hist_msg   = widgets.HTML("")   # Mensaje si falta el archivo
    pval_msg   = widgets.HTML("")

    def _read_binary(p: Path) -> bytes:
        with open(p, "rb") as f:
            return f.read()

    def update(*_):
        feature = feat_dd.value
        split   = split_dd.value

        title_html.value = f"<h4 style='margin:4px 0'>Feature: <code>{feature}</code> &nbsp;|&nbsp; Split: <code>{split}</code></h4>"

        # Rutas
        hist_path = PLOTS_DIR / f"{feature}_hist_all.png"
        pval_path = PLOTS_DIR / f"{feature}_pvalues_{split}.png"

        # Histograma
        if hist_path.exists():
            hist_img.value = _read_binary(hist_path)
            hist_img.format = 'png'
            hist_msg.value = ""
        else:
            hist_img.value = b""
            hist_msg.value = f"<span style='color:#b00'>No se encontró histograma: {hist_path.name}</span>"

        # P-values (según split)
        if pval_path.exists():
            pval_img.value = _read_binary(pval_path)
            pval_img.format = 'png'
            pval_msg.value = ""
        else:
            pval_img.value = b""
            pval_msg.value = f"<span style='color:#b00'>No se encontró p-values: {pval_path.name}</span>"

    # Conectar cambios (si se dispara dos veces, NO duplica nada: solo actualiza el mismo widget)
    feat_dd.observe(update, names="value")
    split_dd.observe(update, names="value")

    # Layout
    controls = widgets.HBox([feat_dd, split_dd])
    hist_box = widgets.VBox([widgets.HTML("<b>Histograma</b>"), hist_img, hist_msg])
    pval_box = widgets.VBox([widgets.HTML("<b>p-values</b>"), pval_img, pval_msg])
    ui = widgets.VBox([controls, title_html, hist_box, pval_box])

    display(ui)
    update()  

def drift_features(csv, col_feature="Feature", col_pvalue="KS_pvalue"):
    df = pd.read_csv(csv)
    df[col_pvalue] = pd.to_numeric(df[col_pvalue], errors="coerce")
    df[col_feature] = df[col_feature].astype(str).str.strip()

    res = (
        df.groupby(col_feature, as_index=False)[col_pvalue]
          .agg(np.nanmean)
          .rename(columns={col_pvalue: "p_value"})
    )
    res["Drift"] = np.where(res["p_value"] < 0.05, "Yes", "No")
    return res

def drift_comparison(csv_test, csv_val, col_feature="Feature", col_pvalue="KS_pvalue"):
    # Procesar ambos periodos
    df_test = drift_features(csv_test, col_feature, col_pvalue)
    df_val  = drift_features(csv_val,  col_feature, col_pvalue)

    # Combinar por feature
    df_comb = pd.merge(
        df_test[[col_feature, "p_value", "Drift"]],
        df_val[[col_feature, "p_value", "Drift"]],
        on=col_feature,
        suffixes=("_Test", "_Val")
    )

    # Crear MultiIndex para encabezados agrupados
    df_comb.columns = pd.MultiIndex.from_tuples([
        ("", col_feature),
        ("Test", "p_value"), ("Test", "Drift"),
        ("Val",  "p_value"), ("Val",  "Drift"),
    ])

    # === Estilo visual (centrado, fondo blanco, sin índice) ===
    styled = (
        df_comb.style
        .set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "center"), ("background-color", "white"), ("color", "black")]},
                {"selector": "td", "props": [("text-align", "center"), ("background-color", "white"), ("color", "black")]},
                {"selector": "tr", "props": [("background-color", "white")]},
                {"selector": "table", "props": [("margin-left", "auto"), ("margin-right", "auto"), ("border-collapse", "collapse")]}
            ]
        )
        .hide(axis="index")  # oculta el índice
    )

    return styled


### Distribution and p_value visualization

In [3]:
feature_image_viewer(plots_dir="drift_simple_report/plots", default_split="val", img_width=1000)



#### Explanation 
In the top is a widget to select which feature and the period to analyze

Histograms: 
* The navy color explains the shape of the distribution in the training data.
* The cornflowerblue color explains the shape of the distribution in the test data.
* The skyblue color explains the shape of the distribution in the validation data.
* The three histograms are overlaid to analyze how similar or different the distributions are. 

P-values timeline:
* The dotted line indicates where is the confidence level threshold.
* The line with circles shows the evolution of the p-values over time.
* The x-axis indicates the dates when the KS test was performed.
* The Y- axis indicates the p-value of the tests.
* The line of p-values helps to understand how the feature behaves over time wheteher it experienced drift, depending if the p_value is below the threshold line.


### Features, P_Values and Detect Drift

In [4]:
drift_comparison(OUT_DIR / "pvalues_test.csv",OUT_DIR / "pvalues_val.csv", "Feature","KS_pvalue")

C:\Users\Alejandro\AppData\Local\Temp\ipykernel_12000\360936608.py:151: FutureWarning: The provided callable <function nanmean at 0x00000167FFE7B740> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(np.nanmean)
C:\Users\Alejandro\AppData\Local\Temp\ipykernel_12000\360936608.py:151: FutureWarning: The provided callable <function nanmean at 0x00000167FFE7B740> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(np.nanmean)


#### Explanation

* In the feature columns, appears each indicator used for the model.
* In the p-values columns is shown the p-values obtained for each future in their respective time period (Time o Validation)
* Similar to the p-value column, the drift column explains if the feature experienced drift, depending  on whether the p-value is lower than the confidence level.
* At the top, appears test and Validation, this refers to each period of time where the Ks test were applied.
* This table provides an easy way to check which fetures presented drift and during which testing period. 

### Top 5 most dfrifted features


In [6]:
top5_test = pd.read_csv("drift_simple_report/top5_test.csv")
top5_val = pd.read_csv("drift_simple_report/top5_val.csv")
Top5_table_graph(top5_test, top5_val, "drift_simple_report/plots", "top5_test.png", "top5_val.png")

HTML(value='\n    <div style="display:flex; justify-content:flex-start; gap:40px; margin-top:10px;">\n        …

HTML(value='\n        <div style="display:flex; justify-content:center; gap:50px; margin-top:20px;">\n        …

#### Explanation

Tables:
* In the feature column, it is shown the top 5 features with most drift in each period.
* In the window column, it appear in how many windows the feature experienced drift.
* The last column shows the p-value of each indicator.
* The table helps to identify which features presented the most drift.

Graph
* The graph shows the 5 features with the highest drift during the test and validation period. 
* The x-axis inidcates the window rate with a p-value lower than confidence level.
* The y-axis indicates the the feature corresponding to each bar. 
* This graph helps to visualize the top 5 of most drifted features. 
